In [57]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/telco-customer.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Handling missing values

In [58]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
df["TotalCharges"].isna().sum()

np.int64(11)

In [59]:
df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)
df["TotalCharges"].isna().sum()

np.int64(0)

### Feature Engineering 

In [60]:
df = df.drop(columns=["customerID"])

In [61]:
(df["tenure"] == 0).sum()

df["AvgMonthlyCharges"] = (df["TotalCharges"] / df["tenure"].replace(0, np.nan))
df["tenure"].isna().sum()
df[["tenure", "AvgMonthlyCharges"]].max()
df["AvgMonthlyCharges"] = df["AvgMonthlyCharges"].fillna(df["MonthlyCharges"])
df["AvgMonthlyCharges"].isna().sum()

np.int64(0)

In [62]:
df["InternetService"].unique()

<StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str

In [63]:
df["HasInternetService"] = (
    df["InternetService"] != "No"
).astype(int)
(df["HasInternetService"] == 1).sum()

np.int64(5517)

In [64]:
services = [
    'PhoneService',
    'MultipleLines',
    'OnlineBackup',
    'DeviceProtection', 
    'TechSupport', 
    'StreamingTV',
    'StreamingMovies'
]
df["NumServices"] = (
    df[services].apply(lambda c: c.eq("Yes")).sum(axis=1)
)

In [65]:
df["ChargesPerServices"] = np.where(
    df["NumServices"] == 0,
    0,
    df['MonthlyCharges'] / df["NumServices"]
)

In [66]:
df["ContractLevel"] = df["Contract"].map({
    "Month-to-month": 0,
    "One year" : 1,
    "Two year" : 2
})
df["ContractLevel"].unique()
df.drop(columns=["Contract"])

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgMonthlyCharges,HasInternetService,NumServices,ChargesPerServices,ContractLevel
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Yes,Electronic check,29.85,29.85,No,29.850000,1,1,29.850000,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,Mailed check,56.95,1889.50,No,55.573529,1,2,28.475000,1
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,Yes,Mailed check,53.85,108.15,Yes,54.075000,1,2,26.925000,0
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,No,Bank transfer (automatic),42.30,1840.75,No,40.905556,1,2,21.150000,1
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Yes,Electronic check,70.70,151.65,Yes,75.825000,1,1,70.700000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,...,Yes,Mailed check,84.80,1990.50,No,82.937500,1,6,14.133333,1
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,...,Yes,Credit card (automatic),103.20,7362.90,No,102.262500,1,6,17.200000,1
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,...,Yes,Electronic check,29.60,346.45,No,31.495455,1,0,0.000000,0
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,...,Yes,Mailed check,74.40,306.60,Yes,76.650000,1,2,37.200000,0


### Features created 
- ContractLevel - Month-to-month=0, One year=1, Two year=2 
- AvgMonthlyCharges = TotalCharges / tenure  
- ChargesPerService - MonthlyCharges / number of services
- HasInternetService - binary (Yes/No)


### Encoding Categorical values

In [67]:
binary_columns = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
]
for col in binary_columns:
    df[col] = df[col].map({'Yes': 1, 'No': 0 })
df.Dependents.isna().sum(
)

np.int64(0)

In [68]:
one_hot_cols = [
    "gender",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "PaymentMethod"
]

df = pd.get_dummies(df, columns=one_hot_cols,drop_first=True, dtype=int)


In [69]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn", "Contract"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print("Train:")
print(y_train.value_counts(normalize=True))

print("\nTest:")
print(y_test.value_counts(normalize=True))

Train:
Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64

Test:
Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


In [70]:
train_indexes = X_train.index
test_indexes = X_test.index
pd.Series(train_indexes).to_csv("../data/processed/train_indexes.csv", index=False)
pd.Series(test_indexes).to_csv("../data/processed/test_indexes.csv", index=False)

In [71]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_cols = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
]
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [72]:
X_train.head()
X_train.max()

SeniorCitizen                              1.000000
Partner                                    1.000000
Dependents                                 1.000000
tenure                                     1.608483
PhoneService                               1.000000
PaperlessBilling                           1.000000
MonthlyCharges                             1.785939
TotalCharges                               2.802714
AvgMonthlyCharges                        121.400000
HasInternetService                         1.000000
NumServices                                7.000000
ChargesPerServices                        76.400000
ContractLevel                              2.000000
gender_Male                                1.000000
MultipleLines_No phone service             1.000000
MultipleLines_Yes                          1.000000
InternetService_Fiber optic                1.000000
InternetService_No                         1.000000
OnlineSecurity_No internet service         1.000000
OnlineSecuri

In [73]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train,
    y_train
)

print(y_train.value_counts())
print(y_train_resampled.value_counts())

Churn
No     4139
Yes    1495
Name: count, dtype: int64
Churn
No     4139
Yes    4139
Name: count, dtype: int64
